# 01 Data Collection

### The Holy Grail of Slay the Spire Run Data
A person in the Slay the Spire reddit community named  <span style="color: lightblue; font-weight:bold;">u/pants555</span> aggregated data from STS internal metric tools across 77 million runs. The link to that is here: https://www.reddit.com/r/slaythespire/comments/jt5y1w/77_million_runs_an_sts_metrics_dump/. This is great because looking at the features, it's very detailed run breakdowns from floor the player made it to before dying or quitting to the specific floor card reward options and which ones were pick and not picked. There's also lots of useful confounders and noisy features that will be adjusted or removed later. 

The entire set is 65GB, broken down across several 7zip files. I am personally not going to dedicate that much storage to this dataset on my personal computer, I don't realistically need that much data to get meaningful results anyways. Maybe in the future...


Instead, I downloaded the sample from the dropbox link provided by OP. It is a 68MB JSON file collection of all the run data gathered from November 2020.

##### Import Statements

In [13]:
import pandas as pd
import json

In [14]:
with open('../raw_data/november.json', 'r') as f:
    raw = json.load(f)

df_runs = pd.json_normalize([record['event'] for record in raw])
print(df_runs.shape)
df_runs.head()

(123436, 50)


,gold_per_floor,floor_reached,playtime,items_purged,score,play_id,local_time,is_ascension_mode,campfire_choices,neow_cost,...,relics_obtained,event_choices,is_beta,boss_relics,items_purged_floors,is_endless,potions_floor_spawned,killed_by,ascension_level,special_seed
0,"[114, 127, 127, 127, 145, 177, 177, 193, 193, ...",50,3610,"[Strike_R, Strike_R, Defend_R, Defend_R]",531,251eb1e0-5bfe-4c74-a86d-2925b198cd9c,20201101005734,False,"[{'data': 'Anger', 'floor': 7, 'key': 'SMITH'}...",TEN_PERCENT_HP_LOSS,...,"[{'floor': 6, 'key': 'Ginger'}, {'floor': 9, '...","[{'damage_healed': 0, 'gold_gain': 0, 'player_...",False,"[{'not_picked': ['Fusion Hammer', 'Black Star'...","[12, 24, 30, 39]",False,"[1, 5, 8, 18, 19, 38]",Awakened One,0,NaN
1,"[118, 135, 145, 145, 165, 165, 240, 240, 264, ...",17,468,[Undo],157,5366608f-19e5-48d5-85d6-3dd822d4bcb0,20201031195733,False,"[{'data': 'Defragment', 'floor': 6, 'key': 'SM...",NONE,...,"[{'floor': 9, 'key': 'Akabeko'}, {'floor': 14,...","[{'damage_healed': 0, 'gold_gain': 0, 'cards_t...",False,[],[11],False,"[1, 5, 14]",NaN,0,NaN
2,"[99, 99]",0,19,[],0,c393b168-681a-461b-ab36-14b12e65789f,20201101005730,False,[],NONE,...,[],[],False,[],[],False,[],NaN,0,NaN
3,"[119, 133, 143, 155, 3, 28, 28, 59, 59, 89, 99...",51,4500,"[Strike_G, Strike_G]",1491,eb57ae10-795b-4236-b195-990eda09c558,20201031165728,True,"[{'data': 'Glass Knife', 'floor': 12, 'key': '...",NONE,...,"[{'floor': 6, 'key': 'Meat on the Bone'}, {'fl...","[{'damage_healed': 0, 'gold_gain': 0, 'player_...",False,"[{'not_picked': ['SacredBark', 'Runic Dome'], ...","[30, 48]",False,"[1, 2, 11, 13, 16, 21, 23, 29, 33, 35, 39, 45,...",NaN,13,NaN
4,"[113, 124, 143, 157, 157, 71, 106, 106, 175, 1...",16,1209,[],122,776b6c74-f409-42da-80c9-5e6db93fdb58,20201101075731,False,"[{'floor': 8, 'key': 'REST'}, {'data': 'Infern...",NONE,...,"[{'floor': 7, 'key': 'WingedGreaves'}, {'floor...","[{'damage_healed': 0, 'gold_gain': 0, 'player_...",False,[],[],False,"[1, 3, 4, 12, 14]",The Guardian,0,NaN


In [ ]:
df_save = df_runs.copy()
for col in df_save.columns:
    if df_save[col].dtype == 'object':
        df_save[col] = df_save[col].apply(lambda x: json.dumps(x) if isinstance(x, (list, dict)) else x)

df_save.to_parquet('../raw_data/runs_pre_clean.parquet', index=False)
print("Saved.")

Saved.


In [16]:
# Inspect a sample card_choices entry
df_runs['card_choices'].dropna().iloc[0]

[{'not_picked': ['Fiend Fire', 'Corruption'],
  'picked': 'Bludgeon',
  'floor': 0},
 {'not_picked': ['Infernal Blade', 'Thunderclap'],
  'picked': 'Flex',
  'floor': 1},
 {'not_picked': ['Flame Barrier', 'Body Slam'], 'picked': 'Anger', 'floor': 2},
 {'not_picked': ['Pommel Strike', 'Perfected Strike'],
  'picked': 'Reckless Charge',
  'floor': 5},
 {'not_picked': ['Thunderclap', 'Disarm', 'Cleave'],
  'picked': 'SKIP',
  'floor': 6},
 {'not_picked': ['Pommel Strike', 'Armaments'], 'picked': 'Flex', 'floor': 8},
 {'not_picked': ['Body Slam+1', 'Twin Strike+1'],
  'picked': 'Anger+1',
  'floor': 10},
 {'not_picked': ['Warcry', 'Bloodletting'],
  'picked': 'Battle Trance',
  'floor': 11},
 {'not_picked': ['Pommel Strike+1', 'Twin Strike+1'],
  'picked': 'Power Through',
  'floor': 14},
 {'not_picked': ['Perfected Strike+1'], 'picked': 'SKIP', 'floor': 18},
 {'not_picked': ['Dark Embrace'], 'picked': 'SKIP', 'floor': 19},
 {'not_picked': ['Uppercut+1'], 'picked': 'SKIP', 'floor': 21},
 {

In [17]:
print('\n'.join(df_runs.columns.tolist()))

gold_per_floor
floor_reached
playtime
items_purged
score
play_id
local_time
is_ascension_mode
campfire_choices
neow_cost
seed_source_timestamp
circlet_count
master_deck
relics
potions_floor_usage
damage_taken
seed_played
potions_obtained
is_trial
path_per_floor
character_chosen
items_purchased
campfire_rested
item_purchase_floors
current_hp_per_floor
gold
neow_bonus
is_prod
is_daily
chose_seed
campfire_upgraded
win_rate
timestamp
path_taken
build_version
purchased_purges
victory
max_hp_per_floor
card_choices
player_experience
relics_obtained
event_choices
is_beta
boss_relics
items_purged_floors
is_endless
potions_floor_spawned
killed_by
ascension_level
special_seed


In [18]:
df_runs['win_rate'].value_counts().head(20)

win_rate
0.0    123216
1.0       220
Name: count, dtype: int64

In [19]:
df_runs['floor_reached'].value_counts().head(50)

floor_reached
16    15965
0     12087
33    10780
51     9022
23     5102
50     4842
6      4770
24     3863
8      3570
25     3005
7      2925
22     2830
10     2779
56     2378
27     2350
55     2264
11     2186
14     2174
12     2091
21     2062
28     1846
13     1817
29     1670
31     1664
1      1589
30     1449
20     1368
5      1268
40     1211
4      1133
41      962
3       912
2       828
39      769
19      763
42      761
18      689
44      629
38      605
17      525
45      498
46      494
48      475
57      456
47      416
37      400
54      337
36      218
15      129
35      126
Name: count, dtype: int64